# Healthcare Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 2: ED Cost Prediction**.

**Goal**: Predict `ed_cost_next3y_usd` for each patient
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular ED cost data with a simple Random Forest regressor.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your-api-key-here",        # Get from your team dashboard
    team_name="your-team-name-here"     # Your exact team name
)

# Load data from PVC paths
print("📂 Loading Healthcare Challenge 2 data...")

# Load ED cost data
train_costs = pd.read_csv("/home/jovyan/shared/datasets/Healthcare/ed_cost_train.csv")
test_costs = pd.read_csv("/home/jovyan/shared/datasets/Healthcare/ed_cost_test.csv")

print(f"✅ Data loaded:")
print(f"   Train costs: {train_costs.shape}")
print(f"   Test costs: {test_costs.shape}")
print(f"   Train columns: {list(train_costs.columns)}")
print(f"   Test columns: {list(test_costs.columns)}")


In [ ]:
# 2. Tabular-Only Baseline Model and Predictions

# From data inspection - ed_cost columns:
# patient_id, primary_chronic, prior_ed_visits_5y, prior_ed_cost_5y_usd, ed_cost_next3y_usd (train only)

# Select numeric features for baseline
cost_features = ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd']
print(f"📊 Using cost prediction features: {cost_features}")

# Prepare training data
X_train = train_costs[cost_features].fillna(0)
y_train = train_costs['ed_cost_next3y_usd']  # Target variable

# Prepare test data
X_test = test_costs[cost_features].fillna(0)

# Train simple Random Forest baseline
print("🤖 Training Random Forest regressor...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# Create submission file (format: patient_id,ed_cost_next3y_usd)
submission_df = pd.DataFrame({
    'patient_id': test_costs['patient_id'],
    'ed_cost_next3y_usd': predictions
})

# Save predictions
submission_df.to_csv("healthcare_challenge2_predictions.csv", index=False)
print(f"✅ Predictions saved: {submission_df.shape[0]} predictions")
print(f"   Preview: {submission_df.head(3)}")
print(f"   Cost range: ${predictions.min():.2f} to ${predictions.max():.2f}")


In [ ]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 2, "healthcare_challenge2_predictions.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 3!")
